# 06 — GNN link predictors on Colab (GPU)

Trains the Phase 6 GNNs (GraphSAGE / GAT, general and pair-focused objectives) with the **same `src/` code**
as the local repo. Everything is written to Google Drive **after each completed unit** (grid config, final
seed, ablation seed, determinism check, out-of-fold seed); a rerun skips completed units, so after a
disconnect just **Runtime → Run all** again.

**Before running:** upload `colab_bundle.tar.gz` to `MyDrive/collab-link-prediction/` and select
**Runtime → Change runtime type → T4 GPU**.

**After it finishes:** download `MyDrive/collab-link-prediction/gnn_results/` and locally run
`python scripts/import_colab_results.py <downloaded folder>` then `python scripts/gnn_analysis.py`.

In [ ]:
# 1. GPU present?
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive and set paths
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/collab-link-prediction")
BUNDLE = DRIVE / "colab_bundle.tar.gz"
OUT = DRIVE / "gnn_results"          # results land here, one folder per completed unit
REPO = Path("/content/collab-link-prediction")
assert BUNDLE.exists(), f"Upload colab_bundle.tar.gz to {DRIVE}"
OUT.mkdir(parents=True, exist_ok=True)
print(BUNDLE, f"{BUNDLE.stat().st_size / 1e6:.1f} MB")

In [ ]:
# 3. Unpack the bundle to local disk (fast I/O) and verify data checksums
import tarfile, json, hashlib, os
if not (REPO / "MANIFEST.json").exists():
    with tarfile.open(BUNDLE) as t:
        t.extractall("/content")
manifest = json.loads((REPO / "MANIFEST.json").read_text())
bad = []
for rel, info in manifest["files"].items():
    h = hashlib.sha256((REPO / rel).read_bytes()).hexdigest()
    if h != info["sha256"]:
        bad.append(rel)
assert not bad, f"checksum mismatch: {bad[:5]}"
print(f"bundle from {manifest['created']}: {len(manifest['files'])} data files verified")
os.chdir(REPO)

In [ ]:
# 4. Install PyG for Colab's PREINSTALLED torch/CUDA (torch is not reinstalled).
# torch_geometric is pure Python; SAGEConv/GATConv need no compiled extensions.
!pip install -q torch_geometric==2.6.1
import importlib, sys, platform
for mod in ["torch_geometric", "lightgbm", "pyarrow", "yaml", "sklearn", "scipy", "networkx", "pandas", "numpy"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:16s} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"{mod:16s} MISSING -> installing"); subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod])
print("python", platform.python_version())

In [ ]:
# 5. Record the exact environment used for training (also rewritten by run_gnn.py at the end)
import torch_geometric, numpy, pandas, json
versions = {"python": platform.python_version(), "torch": torch.__version__, "torch_geometric": torch_geometric.__version__,
            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(), "gpu": torch.cuda.get_device_name(0),
            "numpy": numpy.__version__, "pandas": pandas.__version__}
(OUT / "versions_session.json").write_text(json.dumps(versions, indent=2))
versions

In [ ]:
# 6. Tests (GNN model, decoder, leakage assertion, metrics, protocols) -- must pass before training
!cd /content/collab-link-prediction && python -m pytest -q -p no:cacheprovider tests/test_gnn.py tests/test_metrics.py tests/test_eval.py tests/test_results.py tests/test_heuristics.py

In [ ]:
# 7. Optional speed check (~2-5 min): a reduced run into a scratch folder, to see GPU epoch times.
# !cd /content/collab-link-prediction && python scripts/run_gnn.py --out /content/dry --device cuda --dry

In [ ]:
# 8. Full run: grid -> final 5 seeds -> dot-decoder ablation -> determinism check -> out-of-fold scores.
# Resumable: completed units (folders with meta.json under OUT) are skipped. Log is appended on Drive.
import subprocess, sys
cmd = [sys.executable, "-u", "scripts/run_gnn.py", "--out", str(OUT), "--device", "cuda"]
with open(OUT / "run.log", "a") as logf:
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end=""); logf.write(line); logf.flush()
assert p.wait() == 0, "run_gnn.py failed -- see the log above / run.log on Drive; rerun this cell to resume"

In [ ]:
# 9. What is on Drive now
import pandas as pd
done = {k: len(list(OUT.glob(f"{k}/**/meta.json"))) for k in ["grid", "final", "ablation", "determinism", "oof"]}
print("completed units:", done)
for f in ["gnn_grid.csv", "gnn.csv", "gnn_determinism.csv", "versions.json"]:
    print(f, (OUT / f).exists())
pd.read_csv(OUT / "gnn_determinism.csv") if (OUT / "gnn_determinism.csv").exists() else None

## Next (locally)
1. Download the whole `MyDrive/collab-link-prediction/gnn_results/` folder.
2. `python scripts/import_colab_results.py ~/Downloads/gnn_results` (validates every CSV row against the results schema).
3. `python scripts/gnn_analysis.py` (paired tests, hybrid LightGBM, `results/gnn.md`).